In [ ]:
# test to see if i can link opposite bus stops to each other using their geometry and other data
import polars as pl
import polars.selectors as cs
import polars_st as st
import os

In [ ]:
db_uri = os.environ["DATABASE_URL"]

In [ ]:
query = """
SELECT id,
       source,
       name,
       ST_AsEWKB(geom) as geometry,
       data
FROM static.stop
WHERE source = 'mta_bus'
"""

dtype = pl.Struct([pl.Field("direction", pl.String)])

df_stops: pl.DataFrame = (
    pl.read_database_uri(query, db_uri)
    .with_columns(pl.col("data").str.json_decode(dtype).name.keep())
    .unnest("data")
)
df_stops
# .select("data").row(0)

In [ ]:
query = """
SELECT *
FROM static.route_stop
WHERE source = 'mta_bus'
"""

dtype = pl.Struct([pl.Field("direction", pl.Int64), pl.Field("headsign", pl.String)])

df_route_stops: pl.DataFrame = (
    pl.read_database_uri(query, db_uri)
    .with_columns(pl.col("data").str.json_decode(dtype=dtype))
    .unnest("data")
)
df_route_stops

## Connect opposite-direction bus stops

**Strategy**: For each bus route, pair stops serving direction 0 with stops serving direction 1. Among all candidate pairs (same route, opposite directions), compute the geographic distance and keep the closest match within a threshold. Most cross-street opposite stops are 15–150 m apart; anything beyond ~500 m is unlikely to be a true opposite stop.


In [ ]:
# --- Build route-stop-direction table ---
# df_route_stops columns after unnest: route_id, stop_id, source, stop_sequence, direction, headsign
route_stop_dir = df_route_stops.select(["route_id", "stop_id", "direction"])

# Attach geometry to each (route, stop, direction) row
stops_routed = route_stop_dir.join(
    df_stops.select(pl.exclude("direction")), left_on="stop_id", right_on="id"
)

# --- Separate by direction ---
dir0 = stops_routed.filter(pl.col("direction") == 0).rename(
    {"stop_id": "stop_id_0", "geometry": "geom_0"}
)
dir1 = stops_routed.filter(pl.col("direction") == 1).rename(
    {"stop_id": "stop_id_1", "geometry": "geom_1"}
)

# --- Candidate pairs: same route, opposite directions ---
# One pair can appear multiple times (one row per shared route).
# Select only geometry columns and deduplicate to avoid redundant distance calculations.
candidates = (
    dir0.join(dir1, on="route_id")
    .select(["stop_id_0", "geom_0", "stop_id_1", "geom_1", "route_id"])
    .unique()
    .with_columns(cs.binary().st.to_srid(2263).name.keep())
)

print(f"Candidate pairs (deduplicated): {len(candidates):,}")
candidates.head(3)


In [ ]:
# --- Compute distance between each candidate pair ---
MAX_DIST = 1500  # feet

candidates_dist = candidates.with_columns(
    dist=pl.col("geom_0").st.distance(pl.col("geom_1"))
).filter(pl.col("dist") <= MAX_DIST)

print(f"Pairs within {MAX_DIST}ft threshold: {len(candidates_dist):,}")

# --- For each stop (either direction), pick the nearest opposite stop ---
# Direction-0 stops → nearest direction-1 match
best_for_dir0 = candidates_dist.group_by("stop_id_0", "route_id").agg(
    # Sort stop_id_1 by dist and take the first one
    pl.col("stop_id_1").sort_by("dist").first().alias("opposite_stop_id"),
    # For the distance itself, simple min() is faster and equivalent to sort + first
    pl.min("dist"),
)

# Direction-1 stops → nearest direction-0 match (flip columns, reuse same frame)
best_for_dir1 = (
    candidates_dist.group_by("stop_id_1", "route_id")
    .agg(
        pl.col("stop_id_0").sort_by("dist").first().alias("opposite_stop_id"),
        pl.min("dist"),
    )
    .rename({"stop_id_1": "stop_id_0"})
)

# Combine: if a stop appears in both (it's in dir0 on one route and dir1 on another,
# which shouldn't happen for MTA bus), keep whichever match is closer.
opposite_stops = (
    pl.concat([best_for_dir0, best_for_dir1])
    .sort("dist")
    .unique(subset=["stop_id_0", "route_id"], keep="first")
    .rename({"stop_id_0": "stop_id"})
    .sort("dist")
)

print(f"\nStops with an opposite match: {len(opposite_stops):,}")
opposite_stops.head(10)


In [ ]:
# --- Sanity check: visualize a sample of pairs as lines on a map ---
# Build LINESTRING geometries connecting each matched pair so we can see them in st.explore().

sample_pairs = (
    candidates_dist.join(
        opposite_stops.rename({"stop_id": "stop_id_0"}), on="stop_id_0", how="inner"
    )
    # Keep only the pair that was actually chosen as the best match
    .filter(pl.col("stop_id_1") == pl.col("opposite_stop_id"))
    .sample(n=min(200, len(candidates_dist)), seed=32)
)

# Create a line geometry from geom_0 → geom_1 using WKT construction, then parse
line_df = sample_pairs.with_columns(
    geometry=st.from_wkt(
        pl.format(
            "LINESTRING ({} {}, {} {})",
            pl.col("geom_0").st.x(),
            pl.col("geom_0").st.y(),
            pl.col("geom_1").st.x(),
            pl.col("geom_1").st.y(),
        )
    ).st.set_srid(2263)
).select(["stop_id_0", "stop_id_1", "dist", "geometry", "route_id"])

line_df.st.explore("geometry")


In [ ]:
# --- Quality summary ---
import altair as alt

alt.data_transformers.enable("vegafusion")

n_total_stops = df_route_stops.filter(pl.col("direction").is_not_null()).height
n_matched = len(opposite_stops)

print(f"Total mta_bus stops:        {df_route_stops.height:,}")
print(f"Stops with direction data:  {n_total_stops:,}")
print(
    f"Stops matched to opposite:  {n_matched:,}  ({n_matched / df_route_stops.height * 100:.1f}%)"
)
print()
print("Distance distribution (matched pairs):")
print(opposite_stops.select("dist").describe())

# Histogram of distances
opposite_stops["dist"].plot.hist()

In [ ]:
# see unmatched stops on map
df_stops.join(opposite_stops, how="anti", left_on="id", right_on="stop_id").st.explore()

In [ ]:
# see stops that are far away and might be mismatches
opposite_stops.filter(pl.col("dist") > 1000).join(
    df_stops, left_on="stop_id", right_on="id"
).st.explore()

In [ ]:
# TODO: this is an example of an incorrectly chosen opposite stop id. might be able to fix it by just lowering max dist threshold
opposite_stops.filter(stop_id="305088")